In [ ]:
from skimage import io
import matplotlib.pyplot as plt
import jax.numpy as jnp
from fft import *
import jax

In [ ]:
N = 256

# Follows setup in incoherent CRLB / RL paper
raw_img = io.imread("cameraman.png")[:, :]
min_img = jnp.min(raw_img)
max_img = jnp.max(raw_img)
normalized_img = (raw_img - min_img) / (max_img - min_img)
print("Min and max pixel values from 0 - 255:", jnp.min(raw_img), jnp.max(raw_img))

x = jnp.ones_like(normalized_img)
xhat_goal = jnp.sqrt(raw_img / jnp.mean(raw_img))


def plot_complex_field(x):
    plt.imshow(jnp.abs(x) ** 2)
    plt.colorbar()
    plt.show()
    plt.imshow(jnp.angle(x))
    plt.colorbar()
    plt.show()

In [ ]:
def gerchberg_saxton(amp_src, amp_tgt, random_phase=True):
    print(jnp.sqrt(jnp.mean((amp_src) ** 2)))
    print(jnp.sqrt(jnp.mean((amp_tgt) ** 2)))
    if random_phase:
        key = jax.random.key(0)
        # Uniform random phase in [-pi, pi)
        key, subkey = jax.random.split(key)
        phi = jax.random.uniform(subkey, amp_src.shape, minval=-jnp.pi, maxval=jnp.pi)

        # If you want the corresponding complex unit-modulus field exp(i*phi):
        a = jnp.exp(1j * phi)
    else:
        a = ifft2_shifted(amp_tgt)
    for i in range(1000):
        b = amp_src * jnp.exp(1j * jnp.angle(a))
        c = fft2_shifted(b)
        d = amp_tgt * jnp.exp(1j * jnp.angle(c))
        a = ifft2_shifted(d)

        # Use amplitude difference between c and given amp_tgt as convergence signal
        err_L2 = jnp.sqrt(jnp.mean((jnp.abs(a) - amp_src) ** 2))
        if i % 100 == 0:
            print(err_L2)
    yhat_est = amp_src * jnp.exp(1j * jnp.angle(a))
    return yhat_est


# Find necessary phase assuming uniform amplitude of x
x_complex = gerchberg_saxton(jnp.abs(x), jnp.abs(xhat_goal))

In [ ]:
plot_complex_field(x_complex)

In [ ]:
xhat_final = fft2_shifted(x_complex)
plot_complex_field(xhat_final)